# 03 — MLP Multi-class Classifier (Keras)

Trains the supervised attack-family classifier on the **balanced** training
set produced in notebook 02 and validates on the **untouched** validation set.

Architecture is built by `src.models.build_mlp` so production inference can
load the same shape.


In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from src import config as cfg
from src.models import build_mlp
sns.set_theme(style="whitegrid")
print("TF:", tf.__version__)


## 1. Load data, scaler, label map

In [ ]:
train = pd.read_csv(cfg.BALANCED_DIR / "train_balanced.csv")
val   = pd.read_csv(cfg.PROCESSED_DIR / "val.csv")

scaler = joblib.load(cfg.SCALER_FILE)
with open(cfg.FEATURE_NAMES_FILE) as f: feature_names = json.load(f)
with open(cfg.LABEL_MAP_FILE)    as f: label_map = {int(k):v for k,v in json.load(f).items()}

# Re-encode train labels using the persisted map (so train and val share IDs).
inv_map = {v:k for k,v in label_map.items()}
train = train[train[cfg.LABEL_COL].isin(inv_map)].reset_index(drop=True)
train["_y"] = train[cfg.LABEL_COL].map(inv_map)
val["_y"]   = val[cfg.LABEL_COL].map(inv_map)
val = val.dropna(subset=["_y"]).reset_index(drop=True)

X_train = scaler.transform(train[feature_names].astype(np.float32).values)
y_train = train["_y"].astype(int).values
X_val   = scaler.transform(val[feature_names].astype(np.float32).values)
y_val   = val["_y"].astype(int).values

print("X_train:", X_train.shape, "  X_val:", X_val.shape, "  classes:", len(label_map))


## 2. Build the MLP

In [ ]:
model = build_mlp(
    input_dim=X_train.shape[1],
    n_classes=len(label_map),
    hidden_units=cfg.MLP_CONFIG["hidden_units"],
    dropout=cfg.MLP_CONFIG["dropout"],
    learning_rate=cfg.MLP_CONFIG["learning_rate"],
)
model.summary()


## 3. Train

In [ ]:
cfg.MODELS_DIR.mkdir(parents=True, exist_ok=True)

callbacks = [
    EarlyStopping(monitor="val_accuracy", patience=cfg.MLP_CONFIG["patience"],
                  restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint(filepath=str(cfg.MLP_MODEL_FILE),
                    monitor="val_accuracy", save_best_only=True),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=cfg.MLP_CONFIG["epochs"],
    batch_size=cfg.MLP_CONFIG["batch_size"],
    callbacks=callbacks,
    verbose=2,
)


## 4. Learning curves

In [ ]:
h = history.history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(h["loss"], label="train"); axes[0].plot(h["val_loss"], label="val")
axes[0].set_title("loss"); axes[0].legend()
axes[1].plot(h["accuracy"], label="train"); axes[1].plot(h["val_accuracy"], label="val")
axes[1].set_title("accuracy"); axes[1].legend()
plt.tight_layout(); plt.show()


## 5. Validation report

In [ ]:
from sklearn.metrics import classification_report
y_pred = model.predict(X_val, batch_size=2048, verbose=0).argmax(axis=1)
print(classification_report(
    y_val, y_pred,
    target_names=[label_map[i] for i in range(len(label_map))],
    digits=4, zero_division=0,
))


In [ ]:
# Best model has been checkpointed already; re-save in canonical location.
model.save(cfg.MLP_MODEL_FILE)
print("Saved MLP ->", cfg.MLP_MODEL_FILE)
